# We are comparing the baseline methods with our multi-agent method

## First, we load the questions that GPT-5-nano never got right in all 4 attempts

In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)
from data_processing.data_analysis import select_problem_sample_for_model 
gpt5_nano="GPT-5-nano (high)"
df = select_problem_sample_for_model(gpt5_nano)
df

d:\conda\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Unnamed: 0,source,problem,competition,unique_problem_label,correct,parsed_answer,gold_answer,output_cost_per_tokens,problem_idx,cost,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,input_cost_per_tokens,ten_percentile_group
5836,3928,NaN,Let $N$ denote the numbers of ordered triples ...,MathArena/aime_2025_outputs,MathArena/aime_2025: 15,False,147,735,0.4,15,0.024355,60875.0,107.0,\boxed{147},"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,1
15668,3920,NaN,A plane $\mathcal{P}$ intersects a rectangular...,MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025: 30,False,sqrt(110),\sqrt{\frac{95}{24}},0.4,30,0.017099,42737.0,79.0,Let the six edges of the hexagonal cross-secti...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,2
20160,2292,NaN,Consider all positive multiples of $77$ less t...,MathArena/cmimc_2025_outputs,MathArena/cmimc_2025: 5,False,25,194832,0.4,5,0.002802,6995.0,76.0,### Final answer\n\nReasoning:\n- The positive...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,4
24100,2552,NaN,Let $A B C D E F$ be a convex cyclic hexagon. ...,MathArena/brumo_2025_outputs,MathArena/brumo_2025: 25,False,R,\frac{1+\sqrt{31}}{2},0.4,25,0.011091,27716.0,84.0,- Let the circle’s circumradius be R. For the ...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,6
24116,2568,NaN,"$4$ bears - Aruno, Bruno, Cruno and Druno - ar...",MathArena/brumo_2025_outputs,MathArena/brumo_2025: 21,False,250,160,0.4,21,0.006556,16368.0,181.0,"Step 1: Let a, b, c, d be the four positive in...","Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,8


We have 5 problems that GPT-5 nano couldn't solve in various ranges of difficulty. We'll go from the easiest to hardest.

## Problem 1 - Difficulty level 8

In [2]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==8].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                             MathArena/brumo_2025: 21
answer                  Step 1: Let a, b, c, d be the four positive in...
gold_answer                                                           160
ten_percentile_group                                                    8
problem                 $4$ bears - Aruno, Bruno, Cruno and Druno - ar...
Name: 24116, dtype: object

In [3]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)

$4$ bears - Aruno, Bruno, Cruno and Druno - are each given a card with a
positive integer and are told that the sum of their $4$ numbers is $17$. They
cannot show each other their cards, but discuss a series of observations in the
following order:  Aruno: "I think it is possible that the other three bears all
have the same card." Bruno: "At first, I thought it was possible for the other
three bears to have the same card. Now I know it is impossible for them to have
the same card." Cruno: "I think it is still possible that the other three bears
have the same card." Druno: "I now know what card everyone has." What is the
product of their four card values?
answer 160


### Reflexion attempt

In [4]:
from multi_agent.multi_agent import Problem
from prompt_template import Solver, Reflector
problem = problem = Problem(
    roles=[Solver, Reflector],  # add Solver if you have one
    problem_descr=first_problem_description,
    answer=first_problem_gold_answer
)

print(problem)

Welcome Solver, and Reflector. Together, you should solve the following problem:
>> $4$ bears - Aruno, Bruno, Cruno and Druno - are each given a card with a
positive integer and are told that the sum of their $4$ numbers is $17$. They
cannot show each other their cards, but discuss a series of observations in the
following order:  Aruno: "I think it is possible that the other three bears all
have the same card." Bruno: "At first, I thought it was possible for the other
three bears to have the same card. Now I know it is impossible for them to have
the same card." Cruno: "I think it is still possible that the other three bears
have the same card." Druno: "I now know what card everyone has." What is the
product of their four card values?.<<  "When you are done, you should submidt
your answer as: ANSWER: <your answer>.  No latex formatting, just the raw
number/numbers or strings at the very end.  Before you start sharing your
toughts, give a little summary of the conversation so far.  Giv

In [5]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [12]:
from reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)


=== Attempt 1 ===
Let the four cards be a, b, c, d (Aruno, Bruno, Cruno, Druno) with positive integers and a + b + c + d = 17.

1) From Aruno’s statement:
I think it is possible that the other three bears (B, C, D) all have the same card.
This is possible exactly when there exists a positive integer x with a + 3x = 17, i.e. 17 − a is divisible by 3 and x ≥ 1.
Thus a ∈ {2, 5, 8, 11, 14}.

2) From Bruno’s statement:
Bruno says that before Aruno spoke he thought A=C=D could be equal, but after hearing Aruno he now knows that A=C=D is impossible.
For A=C=D to be possible after Aruno’s statement, there must be some x such that b + 3x = 17 and x ∈ {2, 5, 8, 11, 14} (since Aruno’s a must be in that set). The only b in {1,…,14} for which there exists x ∈ {2,5,8,11,14} with b + 3x = 17 are b ∈ {2, 5, 8, 11, 14}, and specifically:
- If b=2, x=5 works.
- If b=5, x=4 (not in the allowed set).
- If b=8, x=3 (not allowed).
- If b=11, x=2 works.
- If b=14, x=1 (not allowed).

Since Bruno says it is 

In [13]:
print("Number of tokens used", model.num_tokens)

Number of tokens used [CompletionUsage(completion_tokens=16023, prompt_tokens=182, total_tokens=16205, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=14848, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), CompletionUsage(completion_tokens=6559, prompt_tokens=1417, total_tokens=7976, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=6144, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))]


The Reflexion agent solved this problem in 1 attempt

### Tree of thought attempt

In [6]:
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=first_problem_description, answer=first_problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

path d:\NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ("- Step 1: From Aruno's statement, a is such that 17 − a can be written as 3t with t a positive integer. So 17 − a ≡ 0 (mod 3) and 17 − a > 0, giving a ∈ {2, 5, 8, 11, 14}.\n\n- Step 2: Bruno says: initially it was possible for A, C, D to be equal, but after hearing Aruno, it’s now impossible. If A, C, D were equal, then a = c = d = t and 17 − b = 3t. For such a triple to be consistent with Aruno’s constraint a ∈ {2, 5, 8, 11, 14}, t must be in {2, 5, 8, 11, 14}. But 17 − b must also be 3t, so 3t ≤ 16, giving t ∈ {2, 5} and hence b ∈ {11, 2}. Since Bruno now says it’s impossible, b ≠ 2, 11. Together with the initial requirement that 17 − b is divisible by 3 (for the initial possibility), we get b ∈ {5, 8, 14}.\n\n- Step 3: Cruno says it’s still possible that A, B, D are equal. If a = b = d = t, then 17 − c 

The tree of thought starts out with a wrong answer but eventually gets to the right answer after branching a few times.

### Solver-Rejector method - The proposed method

In [6]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

first_problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem1" ,n_steps=10, problem=first_problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

Our method didn't rank the right answer as the highest in this case.

## Problem 2 - Difficulty Level 6

In [7]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==6].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                             MathArena/brumo_2025: 25
answer                  - Let the circle’s circumradius be R. For the ...
gold_answer                                         \frac{1+\sqrt{31}}{2}
ten_percentile_group                                                    6
problem                 Let $A B C D E F$ be a convex cyclic hexagon. ...
Name: 24100, dtype: object

In [ ]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)